# Lecture 5: Document Loader --- Ingesting Data 

- **Course** : NLP with Langchain
___

## The Big Picture 
Imagine you are building an AI assistant for bookstore . Your data lives everywhere:

- Product catalog in a **CSV** file
- Book summaries in **PDF** files
- Customer reviews in a **text** file
- Author bias in **website** 
  

How do you feel ALL of this into your AI assistant?Your need a **Document Loader** to ingest data from various sources and formats into a structured format that your AI can understand and use effectively.


### What you will learn in this lecture:


| # | Topic | Real-world Analogy |
|---|-------|--------------------|
|1  | Document Objects | The "Universal Shipping box" for all data |
|2 | Text & PDF Loaders | Reading books and documents |
|3 | CSV Loaders | Reading spreadsheets with row by row |
|4 | Web Loader | Grabbing into from websites |
|5 | Cloud Loaders | Connecting to google drive You Tube etc.|
|6 | Hands on Experience | Build it yourself! |



___



# 0. Enivronment Setup
Run this cell once to install the package we'll need today.
If you have already installed it, you can skip this step.

In [2]:
# install the required package
%pip install langchain langchain-community pypdf beautifulsoup4 lxml


Note: you may need to restart the kernel to use updated packages.


### data about the data is metadata.


`+-----------------------------------+`

`--------- Document Object -----------`

`--- Page_content  = "actual_text"----`

`----metadata      = {source,page}----`

`+-----------------------------------+`


- page_content (str) = The actual text extracted from file.
- metadata (dict)    = Extra info ... where it can from , page number , author etc.

In [3]:
from langchain_core.documents import Document

# lets create a document manually to see what it looks like
sample_doc = Document(
    page_content = 'Langchain makes it easy to build AI apps',
    metadata = {
        'source': 'manual creation',
        'page': 0,
        'author': 'Hafiz Adnan Sami'

    }
)

# Print what's inside our document
print(f"Content: {sample_doc.page_content}")
print(f"Metadata: {sample_doc.metadata}")
print(f"Type: {type(sample_doc)}")

Content: Langchain makes it easy to build AI apps
Metadata: {'source': 'manual creation', 'page': 0, 'author': 'Hafiz Adnan Sami'}
Type: <class 'langchain_core.documents.base.Document'>


In [4]:
# you can access metadata fields individually as well
source = sample_doc.metadata.get('source', 'unknown ')
page_number = sample_doc.metadata.get('page', -1)
author = sample_doc.metadata.get('author', 'N / A')


print(f"Source: {source}")
print(f"Page Number: {page_number}")
print(f"Author: {author}")


Source: manual creation
Page Number: 0
Author: Hafiz Adnan Sami


# 2. TextLoader --- Reading plain text files
The simplest loader. It reads a text file and wrap it in a Document object.

Think of it as : "Reading a book and putting it in a box for the AI to use later"

**Best for**: Plain text files, logs, transcripts, etc.


In [5]:
from langchain_community.document_loaders import TextLoader

# step1 : create a text file with some content (tell it which files to read)
text_loader = TextLoader(
    file_path=r"E:\NLP\data\nlp.txt",
    encoding='utf-8',
    
)

# step2 : load the document
documents = text_loader.load()

# step3 : print the loaded document
print(f"Loaded documents: {len(documents)}")
print(f"\n--- The Text File Content ---\n")
print(documents[0].page_content)
print(f"\n--- Metadata ---\n")
print(documents[0].metadata)

Loaded documents: 1

--- The Text File Content ---

Introduction to Natural Language Processing
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the interaction between computers and human language. The goal of NLP is to enable machines to understand, interpret, and generate human language in a valuable way.
NLP combines computational linguistics with machine learning and deep learning techniques. It is widely used in applications such as chatbots, machine translation, sentiment analysis, and information retrieval systems.

History of NLP
The history of NLP began in the 1950s with early rule-based systems. One of the first examples was the Georgetown-IBM experiment in , which demonstrated automatic translation of Russian sentences into English.
In the, statistical methods became popular. Researchers started using large corpora of text to train models. In recent years, deep learning approaches such as transformers have significantly improved NLP

# 3. PDFLoader --- Reading PDF page by page
PDFs are more complex than plain text. They can have multiple pages, images, and different layouts. The PDFLoader reads each page and creates a Document object for each one.

**Analogy**: If TextLoader reads a whole notebook at once , PyPDFLoader reads tears out each page and gives it to you one by one.

**Best for**: PDF documents, research papers, reports, etc.

In [6]:
from langchain_community.document_loaders import PyPDFLoader

# step1 : create a PDF loader (tell it which PDF file to read)
pdf_loader = PyPDFLoader(
    file_path=r"E:\NLP\data\Introduction to Natural Language Processing.pdf"
)
# step2 : load the PDF document
pdf_documents = pdf_loader.load()

# step3 : print the loaded PDF documents
print(f"Loaded PDF documents: {len(pdf_documents)}")
print(f"\n--- PDF Content ---\n")

for index , doc in enumerate(pdf_documents):
    print(f"Page {index + 1}:\n{doc.page_content}\n")
    print(f"Metadata: {doc.metadata}\n")


Loaded PDF documents: 3

--- PDF Content ---

Page 1:
Introduction to Natural Language Processing 
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the 
interaction between computers and human language. The goal of NLP is to enable machines to 
understand, interpret, and generate human language in a valuable way. 
NLP combines computational linguistics with machine learning and deep learning techniques. It 
is widely used in applications such as chatbots, machine translation, sentiment analysis, and 
information retrieval systems. 
 
History of NLP 
The history of NLP began in the 1950s with early rule-based systems. One of the first examples 
was the Georgetown-IBM experiment in 1954, which demonstrated automatic translation of 
Russian sentences into English. 
In the 1980s, statistical methods became popular. Researchers started using large corpora of text 
to train models. In recent years, deep learning approaches such as transformers have s

# 4. CSVLoader — Turning Spreadsheet Rows into Documents
This is where it gets interesting! CSVLoader turns each row of your CSV
into a separate Document. Column names become part of the content.

**Analogy**: Imagine a filing cabinet. Each drawer (row) becomes its own document,
with the folder label (column names) included.

Best for: Product catalogs, customer lists, inventory data, survey results.

In [7]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# Load our bookstore product catalog
csv_loader = CSVLoader(
    file_path="data\products.csv",
    encoding="utf-8",
)
csv_documents = csv_loader.load()

print(f"Total rows loaded as documents: {len(csv_documents)}")

print(f"\n--- Sample Document from CSV ---\n")
for index, doc in enumerate(csv_documents[:2]):
    print(f"Document {index + 1}:\n{doc.page_content}\n")
    print(f"Metadata: {doc.metadata}\n")

Total rows loaded as documents: 8

--- Sample Document from CSV ---

Document 1:
product_id: 1
product_name: Python Crash Course
category: Book
price: 29.99
description: A hands-on project-based introduction to programming

Metadata: {'source': 'data\\products.csv', 'row': 0}

Document 2:
product_id: 2
product_name: NLP with Transformers
category: Book
price: 49.99
description: Building language applications with Hugging Face

Metadata: {'source': 'data\\products.csv', 'row': 1}



# 5. WebBaseLoader — Grabbing Data from Websites
What if your data isn't in a file at all — it's on a website?
WebBaseLoader uses BeautifulSoup to scrape web pages and turn them into Documents.

**Analogy**: It's like copying text from a web page and pasting it into a document,
but automated and at scale.

**Best for**: Documentation sites, blog posts, Wikipedia articles, news.
Not great for: JavaScript-heavy sites (SPAs) — the content may not load.

In [8]:
from langchain_community.document_loaders import WebBaseLoader

# Let's load the Wikipedia page about NLP
web_loader = WebBaseLoader(
    web_paths=["https://en.wikipedia.org/wiki/Natural_language_processing"],
)
web_documents = web_loader.load()

print(f"Documents loaded: {len(web_documents)}")
print(f"\n--- Metadata ---")
print(web_documents[0].metadata)

# Web pages can have 50,000+ characters — too much to print at once!
# To see the FULL page content, use: print(web_documents[0].page_content)
print(f"\n--- Content (first 200 chars) ---")
print(web_documents[0].page_content[:200])

Documents loaded: 1

--- Metadata ---
{'source': 'https://en.wikipedia.org/wiki/Natural_language_processing', 'title': 'Natural language processing - Wikipedia', 'language': 'en'}

--- Content (first 200 chars) ---




Natural language processing - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom ar


## MultiPageLoader — Handling Multiple Documents at Once

In [9]:
# You can load MULTIPLE web pages at once!
multi_loader = WebBaseLoader(
    web_paths=[
        "https://en.wikipedia.org/wiki/Machine_learning",
        "https://en.wikipedia.org/wiki/Artificial_intelligence",
    ],
)


multi_docs = multi_loader.load()

for index, doc in enumerate(multi_docs):
    # .get() safely reads a key from the metadata dictionary
    # If the key doesn't exist, it returns the fallback value ("Unknown" / "No Title")
    source = doc.metadata.get("source", "Unknown")
    title = doc.metadata.get("title", "No Title")
    author = doc.metadata.get("author", "N/A")

    # len() counts the total number of characters in the page content
    content_length = len(doc.page_content)

    print(f"\nDocument {index + 1}:")
    print(f"  Title: {title}")
    print(f"  Source: {source}")
    print(f"  Author: {author}")
    # :, inside the f-string adds commas to large numbers (e.g., 85432 → 85,432)
    print(f"  Content Length: {content_length:,} characters")


Document 1:
  Title: Machine learning - Wikipedia
  Source: https://en.wikipedia.org/wiki/Machine_learning
  Author: N/A
  Content Length: 125,238 characters

Document 2:
  Title: Artificial intelligence - Wikipedia
  Source: https://en.wikipedia.org/wiki/Artificial_intelligence
  Author: N/A
  Content Length: 214,018 characters


# 6. SitemapLoader — Crawling Entire Websites
Want to load an entire website, not just one page?
SitemapLoader reads a site's sitemap.xml file (which lists all pages)
and loads each page as a Document.

**Analogy**: WebBaseLoader reads one page of a book. SitemapLoader reads
the table of contents and then loads every chapter.

**Best for**: Building a knowledge base from docs sites, company wikis.
Warning: This can be SLOW for large sites — use filter_urls to limit scope.

In [ ]:
# SitemapLoader — Reference Only (takes too long for class demo)

# IMPORTANT: Jupyter notebooks already run an "event loop" (async engine).
# SitemapLoader uses asyncio internally, which conflicts with Jupyter's loop.
# nest_asyncio patches this so both can work together.
# You MUST run these two lines before using SitemapLoader in a notebook.
# run this once if not installed

%pip install nest_asyncio  
%pip install tqdm

import nest_asyncio
nest_asyncio.apply()  # allows nested event loops — fixes the RuntimeError

from langchain_community.document_loaders.sitemap import SitemapLoader

# filter_urls limits which pages to load (otherwise it loads EVERYTHING)
sitemap_loader = SitemapLoader(
    web_path="https://docs.python.org/3/sitemap.xml",
    filter_urls=["https://docs.python.org/3/tutorial"],
)

sitemap_docs = sitemap_loader.load()
print(f"Total pages loaded: {len(sitemap_docs)}")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Fetching pages: 0it [00:00, ?it/s]

Total pages loaded: 0
SitemapLoader is commented out to save time in class.
Try it at home — remember to uncomment nest_asyncio lines first!
